# Scorecard Generation & Tuning: Medical Procedure Loan Default Risk

**Objective:** Build, validate, and tune a credit scorecard for predicting default risk on medical procedure loans.

**Input Data:**
- Prepared dataset: 252,136 rows × 21 features
- Fair lending compliant (protected classes excluded)
- 11 selected features (IV ≥ 0.02, Min_Bin ≥ 5%)

**Output:**
- WOE-encoded features using scorecardpy
- Logistic regression model with hyperparameter tuning
- Credit scorecard with risk categories
- Model performance evaluation

**FIXES APPLIED (vs original notebook):**
1. Removed the manual `calculate_score()` formula — all scoring now uses `sc.scorecard_ply()` for consistency
2. Single set of parameters (`points0=600, odds0=1/20, pdo=20`) used everywhere
3. Cutoff optimisation runs on actual scorecard scores, not on formula-derived scores
4. Business impact uses empirical default rates from scored data, not a theoretical formula

## 1. Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score
)
import scorecardpy as sc
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", font_scale=0.9)

# Load WOE-encoded data from notebook 3
df = pd.read_csv('../data/model_data.csv')
print(f"Loaded WOE-encoded data: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("Data source: Generated by 3_grouping_and_screening.ipynb (WOE-encoded features)")
print("\nTarget distribution:")
print(df['TARGET'].value_counts())
print("\nClass proportions:")
print(df['TARGET'].value_counts(normalize=True))

In [ ]:
# Data Preprocessing: Handle missing values (all features are WOE-encoded numerics)
df_clean = df.copy()

# Check for missing values
missing_counts = df_clean.isnull().sum()
print("Missing values by feature:")
print(missing_counts[missing_counts > 0])

# Fill missing values with median
for col in df_clean.columns:
    if col != 'TARGET' and df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f"  {col}: filled {missing_counts[col]} NaN with median = {median_val:.6f}")

# Separate features and target
X = df_clean.drop(columns=['TARGET'])
y = df_clean['TARGET']

print(f"\nFeature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")
print(f"Number of features: {X.shape[1]}")
print(f"\nFeature names:")
print(X.columns.tolist())

In [ ]:
# Train-Test Split with Stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("=" * 70)
print("STRATIFIED TRAIN-TEST SPLIT (70% train / 30% test)")
print("=" * 70)
print(f"\nTraining set: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:     {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

print(f"\nTraining set class distribution:")
train_dist = y_train.value_counts(normalize=True).sort_index()
for label, prop in train_dist.items():
    count = y_train.value_counts()[label]
    print(f"  Class {label}: {count:>7,} samples ({prop*100:5.2f}%)")

print(f"\nTest set class distribution:")
test_dist = y_test.value_counts(normalize=True).sort_index()
for label, prop in test_dist.items():
    count = y_test.value_counts()[label]
    print(f"  Class {label}: {count:>7,} samples ({prop*100:5.2f}%)")

print(f"\nStratification check (proportions match):")
print(f"  Original:  {y.value_counts(normalize=True).sort_index().values}")
print(f"  Train:     {train_dist.values}")
print(f"  Test:      {test_dist.values}")

## 2. Train Logistic Regression Model

In [ ]:
def train_and_evaluate_model(X_tr, X_te, y_tr, y_te, feature_names, iteration_name):
    print(iteration_name)
    print("=" * 70)
    print(f"Number of features: {X_tr.shape[1]}")
    
    model = LogisticRegression(random_state=42, max_iter=1000, solver='liblinear', class_weight='balanced')
    model.fit(X_tr, y_tr)
    
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    y_tr_pred_proba = model.predict_proba(X_tr)[:, 1]
    y_te_pred_proba = model.predict_proba(X_te)[:, 1]
    
    train_auc = roc_auc_score(y_tr, y_tr_pred_proba)
    test_auc = roc_auc_score(y_te, y_te_pred_proba)
    train_f1 = f1_score(y_tr, y_tr_pred)
    test_f1 = f1_score(y_te, y_te_pred)
    train_acc = accuracy_score(y_tr, y_tr_pred)
    test_acc = accuracy_score(y_te, y_te_pred)
    cm_test = confusion_matrix(y_te, y_te_pred)
    
    print(f"Train: AUC={train_auc:.4f} F1={train_f1:.4f} Acc={train_acc:.4f}")
    print(f"Test:  AUC={test_auc:.4f} F1={test_f1:.4f} Acc={test_acc:.4f}")
    print(f"Confusion Matrix - TN:{cm_test[0,0]} FP:{cm_test[0,1]} FN:{cm_test[1,0]} TP:{cm_test[1,1]}")
    print("Classification Report (Test):")
    print(classification_report(y_te, y_te_pred, target_names=['Non-Default', 'Default']))
    
    coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': model.coef_[0]})
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
                xticklabels=['Non-Default', 'Default'],
                yticklabels=['Non-Default', 'Default'])
    axes[0].set_title(f'{iteration_name} - Confusion Matrix', fontweight='bold')
    axes[0].set_ylabel('Actual', fontweight='bold')
    axes[0].set_xlabel('Predicted', fontweight='bold')
    
    fpr_train, tpr_train, _ = roc_curve(y_tr, y_tr_pred_proba)
    fpr_test, tpr_test, _ = roc_curve(y_te, y_te_pred_proba)
    axes[1].plot(fpr_train, tpr_train, 'b-', linewidth=2.5, label=f'Train (AUC={train_auc:.4f})')
    axes[1].plot(fpr_test, tpr_test, 'r-', linewidth=2.5, label=f'Test (AUC={test_auc:.4f})')
    axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random')
    axes[1].set_xlabel('False Positive Rate', fontweight='bold')
    axes[1].set_ylabel('True Positive Rate', fontweight='bold')
    axes[1].set_title(f'{iteration_name} - ROC Curve', fontweight='bold')
    axes[1].legend(loc='lower right', fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim([0, 1])
    axes[1].set_ylim([0, 1])
    
    plt.tight_layout()
    plt.show()
    
    return {
        'model': model,
        'features': list(feature_names),
        'coefficients': coef_df,
        'train_metrics': {'auc': train_auc, 'f1': train_f1, 'accuracy': train_acc},
        'test_metrics': {'auc': test_auc, 'f1': test_f1, 'accuracy': test_acc},
        'y_train_pred_proba': y_tr_pred_proba,
        'y_test_pred_proba': y_te_pred_proba,
        'confusion_matrix': cm_test
    }

print("Ready")

### Iteration 1: All features

In [ ]:
iter1_features = X.columns
iter1_results = train_and_evaluate_model(X_train, X_test, y_train, y_test, iter1_features, "ITERATION 1: All 13 Features")

### Model Iteration 2: Top-10 features by coefficient

In [ ]:
top_10_features = iter1_results['coefficients'].sort_values('Coefficient', key=abs, ascending=False).head(10)['Feature'].tolist()
X_train_iter2 = X_train[top_10_features]
X_test_iter2 = X_test[top_10_features]
iter2_results = train_and_evaluate_model(X_train_iter2, X_test_iter2, y_train, y_test, top_10_features, "ITERATION 2: Top 10 Features")

### Iteration 3: By Domain knowledge

In [ ]:
domain_features = ['debt_to_credit_woe', 'avg_days_credit_woe', 'DAYS_EMPLOYED_woe', 'num_active_woe', 'DAYS_LAST_PHONE_CHANGE_woe', 'NAME_INCOME_TYPE_woe', 'NAME_EDUCATION_TYPE_woe', 'ORGANIZATION_TYPE_woe', 'REGION_RATING_CLIENT_W_CITY_woe']
X_train_iter3 = X_train[domain_features]
X_test_iter3 = X_test[domain_features]
iter3_results = train_and_evaluate_model(X_train_iter3, X_test_iter3, y_train, y_test, domain_features, "ITERATION 3: Domain Knowledge (9 Features)")

### Overall comparison

In [ ]:
comp_data = {
    'Iteration': ['Iter 1 (13)', 'Iter 2 (10)', 'Iter 3 (9)'],
    'Test_AUC': [iter1_results['test_metrics']['auc'], iter2_results['test_metrics']['auc'], iter3_results['test_metrics']['auc']],
    'Test_F1': [iter1_results['test_metrics']['f1'], iter2_results['test_metrics']['f1'], iter3_results['test_metrics']['f1']]
}
comp_df = pd.DataFrame(comp_data)
print("ITERATION COMPARISON")
print(comp_df.to_string(index=False))

In [ ]:
best_idx = comp_df["Test_AUC"].idxmax()
models = [iter1_results, iter2_results, iter3_results]
best_model_results = models[best_idx]

print(f"\nBest Model: Iteration {best_idx + 1}")
print(f"Test AUC: {best_model_results['test_metrics']['auc']:.4f}")
print(f"Test F1: {best_model_results['test_metrics']['f1']:.4f}")

# Store best model variables for Section 3
selected_model = best_model_results['model']
selected_features = best_model_results['features']

print(f"\nBest model stored with {len(selected_features)} features")
print(f"Selected features: {selected_features}")

## 3. Scorecard Generation (scorecardpy — single consistent system)

**KEY DESIGN DECISION:** All scoring uses `scorecardpy` exclusively.  
- `sc.scorecard()` generates the point-allocation table  
- `sc.scorecard_ply()` scores every applicant  
- Cutoff optimisation runs on those scores  

No manual `calculate_score()` formula — that was the source of the original bug  
(it used `points0=600` while the scorecard used `points0=640`, and the odds direction was inverted).

In [ ]:
# ============================================================
# 3.1 LOAD BINS & RAW DATA
# ============================================================
print("=" * 70)
print("SECTION 3: SCORECARD GENERATION AND TUNING")
print("=" * 70)

# Load bins from notebook 3
pkl_path = Path("../outputs/bins_dict.pkl")
if pkl_path.exists():
    with open(pkl_path, 'rb') as f:
        bins_dict_lists = pickle.load(f)
    bins = {variable: pd.DataFrame(bl) for variable, bl in bins_dict_lists.items()}
    print(f"[OK] Loaded bins_dict with {len(bins)} variables")
else:
    raise FileNotFoundError(f"bins_dict.pkl not found at {pkl_path.absolute()}. Run notebook 3 first.")

# Load raw (pre-WOE) data for sc.scorecard_ply()
# scorecard_ply needs the ORIGINAL feature values, not WOE-encoded ones
raw_path = Path("../data/prepared_data.csv")
if raw_path.exists():
    df_raw = pd.read_csv(raw_path)
    print(f"[OK] Loaded raw data: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
else:
    raise FileNotFoundError(f"Raw data not found at {raw_path.absolute()}. Need pre-WOE data for scoring.")

# Derive the raw feature names (strip _woe suffix from selected_features)
raw_feature_names = [f.replace('_woe', '') for f in selected_features]
print(f"\nRaw feature names for scorecard: {raw_feature_names}")

# Build raw train/test aligned with the WOE train/test split
# Use the same indices from the stratified split
raw_features_with_target = raw_feature_names + ['TARGET']
df_raw_subset = df_raw[raw_features_with_target].copy()

# Re-split using the SAME indices
X_raw_train = df_raw_subset.loc[X_train.index, raw_feature_names].copy()
X_raw_test = df_raw_subset.loc[X_test.index, raw_feature_names].copy()
print(f"\nRaw train shape: {X_raw_train.shape}")
print(f"Raw test shape:  {X_raw_test.shape}")

In [ ]:
# ============================================================
# 3.2 GENERATE SCORECARD
# ============================================================
print("\n" + "=" * 70)
print("CREDIT SCORECARD GENERATION")
print("=" * 70)

# --- SINGLE SET OF PARAMETERS (used EVERYWHERE) ---
points0 = 600        # Base score
odds0   = 1 / 20     # At base score: 1 good per 20 bad (odds = 0.05)
pdo     = 20         # Points to double the odds

print(f"\nScorecard Parameters (used consistently throughout):")
print(f"  Base Score (points0): {points0}")
print(f"  Base Odds (odds0):    {odds0} (1:{int(1/odds0)})")
print(f"  PDO:                  {pdo}")

# Generate scorecard
card = sc.scorecard(
    bins,
    selected_model,
    selected_features,
    points0=points0,
    odds0=odds0,
    pdo=pdo,
    basepoints_eq0=False
)

print("\n[SUCCESS] Scorecard generated using scorecardpy")

# Convert card to DataFrame for display
scorecard_data = []
for variable, card_df in card.items():
    if variable != 'basepoints':
        for _, row in card_df.iterrows():
            scorecard_data.append({
                "Variable": row['variable'],
                "Bin": row['bin'],
                "Points": row['points']
            })

scorecard_df = pd.DataFrame(scorecard_data)
print(f"Scorecard complete: {len(scorecard_df)} total rows across {scorecard_df['Variable'].nunique()} variables")

# Show max points per variable and theoretical score range
print("\n" + "-" * 70)
print("SCORECARD POINT RANGES PER VARIABLE")
print("-" * 70)
total_min = 0
total_max = 0
for var in scorecard_df['Variable'].unique():
    var_pts = scorecard_df[scorecard_df['Variable'] == var]['Points']
    vmin, vmax = var_pts.min(), var_pts.max()
    total_min += vmin
    total_max += vmax
    print(f"  {var:40s}  min={vmin:6.0f}   max={vmax:6.0f}   spread={vmax-vmin:6.0f}")

print(f"\n  {'TOTAL':40s}  min={total_min:6.0f}   max={total_max:6.0f}   spread={total_max-total_min:6.0f}")
print(f"\n  Theoretical score range: [{total_min:.0f}, {total_max:.0f}]")

In [ ]:
# Display the full scorecard table
print("\nFULL CREDIT SCORECARD")
print("=" * 70)
display(scorecard_df)

In [ ]:
# ============================================================
# 3.3 SCORE ALL APPLICANTS USING sc.scorecard_ply()
# ============================================================
print("\n" + "=" * 70)
print("SCORING ALL APPLICANTS (via sc.scorecard_ply)")
print("=" * 70)

# Score training and test sets using the scorecard
train_scores_df = sc.scorecard_ply(X_raw_train, card, only_total_score=True, print_step=0)
test_scores_df  = sc.scorecard_ply(X_raw_test, card, only_total_score=True, print_step=0)

train_scores = train_scores_df['score'].values
test_scores  = test_scores_df['score'].values

print(f"\nScorecard Score Statistics:")
print(f"  Train - Mean: {train_scores.mean():.2f}, Std: {train_scores.std():.2f}, "
      f"Min: {train_scores.min():.2f}, Max: {train_scores.max():.2f}")
print(f"  Test  - Mean: {test_scores.mean():.2f}, Std: {test_scores.std():.2f}, "
      f"Min: {test_scores.min():.2f}, Max: {test_scores.max():.2f}")

In [ ]:
# ============================================================
# 3.4 SCORE DISTRIBUTION — ENTIRE POPULATION
# ============================================================
combined_scores = np.concatenate([train_scores, test_scores])
combined_target = pd.concat([y_train, y_test]).values

print("\n" + "=" * 70)
print("SCORE DISTRIBUTION — ENTIRE POPULATION")
print("=" * 70)
print(f"\nCombined Score Statistics:")
print(f"  Total observations: {len(combined_scores):,}")
print(f"  Mean:   {combined_scores.mean():.2f}")
print(f"  Std:    {combined_scores.std():.2f}")
print(f"  Min:    {combined_scores.min():.2f}")
print(f"  Max:    {combined_scores.max():.2f}")
print(f"  Median: {np.median(combined_scores):.2f}")
print(f"  P25:    {np.percentile(combined_scores, 25):.2f}")
print(f"  P75:    {np.percentile(combined_scores, 75):.2f}")

# Distribution plot — split by good/bad
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Overall distribution
axes[0].hist(combined_scores, bins=60, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(combined_scores.mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean: {combined_scores.mean():.0f}')
axes[0].axvline(np.median(combined_scores), color='green', linestyle='--', linewidth=2,
                label=f'Median: {np.median(combined_scores):.0f}')
axes[0].set_xlabel('Credit Score', fontweight='bold')
axes[0].set_ylabel('Frequency', fontweight='bold')
axes[0].set_title('Score Distribution — All Applicants', fontweight='bold')
axes[0].legend()

# Split by default status
good_scores = combined_scores[combined_target == 0]
bad_scores  = combined_scores[combined_target == 1]
axes[1].hist(good_scores, bins=50, alpha=0.6, color='steelblue', label=f'Non-Default (n={len(good_scores):,})', density=True)
axes[1].hist(bad_scores, bins=50, alpha=0.6, color='red', label=f'Default (n={len(bad_scores):,})', density=True)
axes[1].set_xlabel('Credit Score', fontweight='bold')
axes[1].set_ylabel('Density', fontweight='bold')
axes[1].set_title('Score Distribution — Good vs Bad', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Cutoff Optimisation (on scorecard scores)

In [ ]:
# ============================================================
# 4.1 CUTOFF OPTIMISATION — EMPIRICAL DEFAULT RATES
# ============================================================
print("\n" + "=" * 70)
print("CUTOFF OPTIMISATION (Empirical — using actual scorecard scores)")
print("=" * 70)

print(f"\nScore Statistics (Test Set):")
print(f"  Score Range: {test_scores.min():.2f} to {test_scores.max():.2f}")
print(f"  Mean: {test_scores.mean():.2f}")
print(f"  Std Dev: {test_scores.std():.2f}")

# Build cutoff range from actual score distribution
score_min = int(np.floor(test_scores.min() / 10) * 10)
score_max = int(np.ceil(test_scores.max() / 10) * 10) + 10
cutoffs = list(range(score_min, score_max, 5))

print(f"  Evaluating cutoffs from {cutoffs[0]} to {cutoffs[-1]} (step=5)")

In [ ]:
# ============================================================
# 4.2 BUSINESS IMPACT ANALYSIS
# ============================================================
print("\n" + "=" * 70)
print("BUSINESS IMPACT ANALYSIS")
print("=" * 70)

# Business parameters
avg_revenue_per_loan  = 1500
avg_loss_per_default  = 10000
total_loans_applied   = 10000

print(f"\nBusiness Parameters:")
print(f"  Total loan applications: {total_loans_applied:,}")
print(f"  Average revenue per loan: ${avg_revenue_per_loan:,}")
print(f"  Average loss per default: ${avg_loss_per_default:,}")

# Use the TEST SET to compute empirical default rates at each cutoff
y_test_arr = y_test.values

results = []
for cutoff in cutoffs:
    # In scorecardpy: HIGHER score = LOWER risk (good)
    # Approve applicants with score >= cutoff
    approved_mask = test_scores >= cutoff
    n_approved = approved_mask.sum()
    approval_rate = n_approved / len(test_scores)
    
    # Empirical default rate among approved applicants
    if n_approved > 0:
        defaults_among_approved = y_test_arr[approved_mask].sum()
        empirical_default_rate = defaults_among_approved / n_approved
    else:
        defaults_among_approved = 0
        empirical_default_rate = 0.0
    
    # Scale to business scenario
    approved_loans     = int(total_loans_applied * approval_rate)
    expected_defaults  = int(approved_loans * empirical_default_rate)
    total_revenue      = approved_loans * avg_revenue_per_loan
    total_loss         = expected_defaults * avg_loss_per_default
    net_profit         = total_revenue - total_loss
    profit_per_loan    = net_profit / total_loans_applied if total_loans_applied > 0 else 0
    
    results.append({
        "Cutoff": cutoff,
        "Approval_Rate": approval_rate,
        "Default_Rate": empirical_default_rate,
        "Approved_Loans": approved_loans,
        "Expected_Defaults": expected_defaults,
        "Revenue": total_revenue,
        "Loss": total_loss,
        "Net_Profit": net_profit,
        "Profit_Per_Loan": profit_per_loan
    })

business_df = pd.DataFrame(results)

# Print summary for key cutoffs
print("\nBusiness Impact by Cutoff (empirical default rates from test set):")
print(f"{'Cutoff':>7} | {'Appr%':>7} | {'DefRate':>7} | {'Approved':>9} | {'Defaults':>9} | {'Revenue':>12} | {'Loss':>12} | {'Profit':>12}")
print("-" * 100)
for _, row in business_df.iterrows():
    print(f"{row['Cutoff']:>7.0f} | {row['Approval_Rate']*100:>6.1f}% | {row['Default_Rate']*100:>6.2f}% | "
          f"{row['Approved_Loans']:>9,.0f} | {row['Expected_Defaults']:>9,.0f} | "
          f"${row['Revenue']:>11,.0f} | ${row['Loss']:>11,.0f} | ${row['Net_Profit']:>11,.0f}")

In [ ]:
# ============================================================
# 4.3 OPTIMAL CUTOFF
# ============================================================
optimal_idx = business_df["Net_Profit"].idxmax()
opt_row = business_df.loc[optimal_idx]

print("\n" + "=" * 70)
print(f"OPTIMAL CUTOFF (Maximum Profit): {opt_row['Cutoff']:.0f}")
print("=" * 70)
print(f"  Approval Rate:    {opt_row['Approval_Rate']*100:.1f}%")
print(f"  Default Rate:     {opt_row['Default_Rate']*100:.2f}%")
print(f"  Approved Loans:   {opt_row['Approved_Loans']:,.0f}")
print(f"  Expected Defaults:{opt_row['Expected_Defaults']:,.0f}")
print(f"  Net Profit:       ${opt_row['Net_Profit']:,.0f}")
print(f"  Profit/Loan:      ${opt_row['Profit_Per_Loan']:,.2f}")

# Store the optimal cutoff for later use
OPTIMAL_CUTOFF = opt_row['Cutoff']

# Sanity check: scorecard range vs cutoff
print(f"\n  Scorecard Score Range (test): [{test_scores.min():.0f}, {test_scores.max():.0f}]")
print(f"  Headroom above cutoff:  {test_scores.max() - OPTIMAL_CUTOFF:.0f} points")
print(f"  Headroom below cutoff:  {OPTIMAL_CUTOFF - test_scores.min():.0f} points")

In [ ]:
# ============================================================
# 4.4 FORMATTED BUSINESS TABLE
# ============================================================
display_df = business_df.copy()

format_mapping = {
    "Cutoff": "{:.0f}",
    "Approval_Rate": "{:.2%}",
    "Default_Rate": "{:.2%}",
    "Approved_Loans": "{:,.0f}",
    "Expected_Defaults": "{:,.0f}",
    "Revenue": "${:,.0f}",
    "Loss": "${:,.0f}",
    "Net_Profit": "${:,.0f}",
    "Profit_Per_Loan": "${:,.2f}"
}

print("BUSINESS IMPACT ANALYSIS TABLE")
display(display_df.style.format(format_mapping).highlight_max(subset=['Net_Profit'], color='lightgreen'))

In [ ]:
# ============================================================
# 4.5 VISUALISATION — Profit & Approval Rate by Cutoff
# ============================================================
fig, ax1 = plt.subplots(figsize=(12, 6))

# Left axis: Profit
color1 = 'steelblue'
ax1.set_xlabel('Credit Score Cutoff', fontsize=12, fontweight='bold')
ax1.set_ylabel('Profit (Millions $)', fontsize=12, fontweight='bold', color=color1)
line1 = ax1.plot(business_df['Cutoff'], business_df['Net_Profit'] / 1e6,
                 color=color1, marker='o', linewidth=2.5, markersize=5, label='Profit')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, alpha=0.3)

# Right axis: Approval Rate
ax2 = ax1.twinx()
color2 = 'red'
ax2.set_ylabel('Approval Rate (%)', fontsize=12, fontweight='bold', color=color2)
line2 = ax2.plot(business_df['Cutoff'], business_df['Approval_Rate'] * 100,
                 color=color2, marker='s', linewidth=2.5, markersize=5, label='Approval Rate', linestyle='--')
ax2.tick_params(axis='y', labelcolor=color2)
ax2.set_ylim([0, 105])

# Mark optimal cutoff
ax1.axvline(x=OPTIMAL_CUTOFF, color='purple', linestyle=':', linewidth=2.5, alpha=0.7)

plt.title('Profit & Approval Rate by Credit Score Cutoff', fontsize=14, fontweight='bold', pad=20)
lines = line1 + line2 + [plt.Line2D([0], [0], color='purple', linestyle=':', linewidth=2.5)]
labels = ['Profit', 'Approval Rate', f'Optimal Cutoff: {OPTIMAL_CUTOFF:.0f}']
ax1.legend(lines, labels, loc='upper right', fontsize=11, frameon=True, shadow=True)

plt.tight_layout()
plt.show()

print(f"\nOptimal Cutoff: {OPTIMAL_CUTOFF:.0f}")
print(f"Profit at optimum: ${opt_row['Net_Profit']/1e6:.2f}M")

## 5. Scorecard Validation

In [ ]:
# ============================================================
# 5.1 SAMPLE APPLICANT — BEST CASE (highest-point bins)
# ============================================================
print("=" * 70)
print("SAMPLE APPLICANT — BEST CASE")
print("=" * 70)

sample_data = {
    'avg_days_credit': [-1500],
    'debt_to_credit': [0.1],
    'DAYS_EMPLOYED': [5000],
    'OCCUPATION_TYPE': ['Accountants'],
    'AMT_GOODS_PRICE': [1500000],
    'NAME_EDUCATION_TYPE': ['Higher education'],
    'REGION_RATING_CLIENT_W_CITY': [1],
    'DAYS_LAST_PHONE_CHANGE': [2500],
    'num_active': [0],
    'ORGANIZATION_TYPE': ['Trade: type 4'],
    'NAME_INCOME_TYPE': ['State servant'],
    'EMERGENCYSTATE_MODE': ['No'],
    'HOUSETYPE_MODE': ['block of flats']
}

sample_df = pd.DataFrame(sample_data)
sample_score = sc.scorecard_ply(sample_df, card, print_step=0)
score_breakdown = sc.scorecard_ply(sample_df, card, only_total_score=False, print_step=0)

print(f"\nSAMPLE APPLICANT SCORE: {sample_score.iloc[0,0]:.0f}")
print(f"Decision: {'APPROVE' if sample_score.iloc[0,0] >= OPTIMAL_CUTOFF else 'DECLINE'} (cutoff = {OPTIMAL_CUTOFF:.0f})")
print(f"\nPoint Breakdown:")
print(score_breakdown.T)

In [ ]:
# ============================================================
# 5.2 SAMPLE APPLICANT — WORST CASE (lowest-point bins)
# ============================================================
print("=" * 70)
print("SAMPLE APPLICANT — WORST CASE")
print("=" * 70)

negative_sample_data = {
    'avg_days_credit': [-100],
    'debt_to_credit': [0.95],
    'DAYS_EMPLOYED': [500],
    'OCCUPATION_TYPE': ['Drivers'],
    'AMT_GOODS_PRICE': [400000],
    'NAME_EDUCATION_TYPE': ['Incomplete higher'],
    'REGION_RATING_CLIENT_W_CITY': [3],
    'DAYS_LAST_PHONE_CHANGE': [100],
    'num_active': [6],
    'ORGANIZATION_TYPE': ['Cleaning'],
    'NAME_INCOME_TYPE': ['Working'],
    'EMERGENCYSTATE_MODE': ['Yes'],
    'HOUSETYPE_MODE': ['terraced house']
}

sample_df_neg = pd.DataFrame(negative_sample_data)
sample_score_neg = sc.scorecard_ply(sample_df_neg, card, print_step=0)
score_breakdown_neg = sc.scorecard_ply(sample_df_neg, card, only_total_score=False, print_step=0)

print(f"\nSAMPLE APPLICANT SCORE: {sample_score_neg.iloc[0,0]:.0f}")
print(f"Decision: {'APPROVE' if sample_score_neg.iloc[0,0] >= OPTIMAL_CUTOFF else 'DECLINE'} (cutoff = {OPTIMAL_CUTOFF:.0f})")
print(f"\nPoint Breakdown:")
print(score_breakdown_neg.T)

In [ ]:
# ============================================================
# 5.3 VALIDATION — Scorecard vs Model Predictions (Rank Ordering)
# ============================================================
print("\n" + "=" * 70)
print("VALIDATION: Scorecard Score vs Model Probability")
print("=" * 70)

# The scorecard scores should be monotonically related to model probabilities
# Higher score (from scorecard) = lower P(default) from model
y_test_pred_proba = selected_model.predict_proba(X_test[selected_features])[:, 1]

from scipy.stats import spearmanr
corr, pval = spearmanr(test_scores, y_test_pred_proba)
print(f"\nSpearman correlation (scorecard score vs P(default)): {corr:.4f}")
print(f"  p-value: {pval:.2e}")
print(f"  Expected: strong NEGATIVE correlation (higher score = lower default risk)")

# Rank-order check: AUC of scorecard scores as a predictor
# For AUC, we need P(bad) ordering, so we negate the score
scorecard_auc = roc_auc_score(y_test, -test_scores)  # negate because higher score = good
model_auc = roc_auc_score(y_test, y_test_pred_proba)

print(f"\nAUC comparison:")
print(f"  Model AUC (continuous):    {model_auc:.4f}")
print(f"  Scorecard AUC (binned):    {scorecard_auc:.4f}")
print(f"  AUC loss from binning:     {model_auc - scorecard_auc:.4f}")

# Spot-check a few samples
print(f"\nSpot-check (5 random test samples):")
print(f"{'Index':>8} | {'Score':>8} | {'P(default)':>10} | {'Actual':>6}")
print("-" * 45)
np.random.seed(42)
check_idx = np.random.choice(len(test_scores), 5, replace=False)
for i in check_idx:
    print(f"{i:>8} | {test_scores[i]:>8.0f} | {y_test_pred_proba[i]:>10.4f} | {y_test.iloc[i]:>6}")

In [ ]:
# ============================================================
# 5.4 SCORE DISTRIBUTION BY DECILE
# ============================================================
print("\n" + "=" * 70)
print("SCORECARD PERFORMANCE BY SCORE DECILE")
print("=" * 70)

test_analysis = pd.DataFrame({
    'score': test_scores,
    'target': y_test.values
})

test_analysis['decile'] = pd.qcut(test_analysis['score'], 10, labels=False, duplicates='drop')

decile_stats = test_analysis.groupby('decile').agg(
    count=('target', 'count'),
    defaults=('target', 'sum'),
    min_score=('score', 'min'),
    max_score=('score', 'max'),
    mean_score=('score', 'mean')
).reset_index()

decile_stats['default_rate'] = decile_stats['defaults'] / decile_stats['count']
decile_stats['pct_population'] = decile_stats['count'] / decile_stats['count'].sum() * 100

print(f"\n{'Decile':>7} | {'Count':>7} | {'Defaults':>9} | {'DefRate':>8} | {'Min':>7} | {'Max':>7} | {'Mean':>7}")
print("-" * 75)
for _, row in decile_stats.iterrows():
    print(f"{int(row['decile']):>7} | {int(row['count']):>7,} | {int(row['defaults']):>9,} | "
          f"{row['default_rate']*100:>7.2f}% | {row['min_score']:>7.0f} | {row['max_score']:>7.0f} | {row['mean_score']:>7.0f}")

# Check monotonicity
rates = decile_stats['default_rate'].values
is_monotonic = all(rates[i] >= rates[i+1] for i in range(len(rates)-1))
print(f"\nDefault rate monotonically decreasing across deciles: {is_monotonic}")
if not is_monotonic:
    print("  (Minor inversions are acceptable due to binning discretisation)")

## 6. Final Summary

In [ ]:
# ============================================================
# 6.1 FINAL SUMMARY
# ============================================================
print("\n" + "=" * 70)
print("FINAL SCORECARD SUMMARY")
print("=" * 70)

print(f"\n--- Model ---")
print(f"  Algorithm:        Logistic Regression (balanced, liblinear)")
print(f"  Features:         {len(selected_features)}")
print(f"  Test AUC (model): {best_model_results['test_metrics']['auc']:.4f}")

print(f"\n--- Scorecard Parameters ---")
print(f"  Base Score:  {points0}")
print(f"  Base Odds:   1:{int(1/odds0)} (odds0 = {odds0})")
print(f"  PDO:         {pdo}")

print(f"\n--- Score Distribution (Test Set) ---")
print(f"  Min:    {test_scores.min():.0f}")
print(f"  Mean:   {test_scores.mean():.0f}")
print(f"  Max:    {test_scores.max():.0f}")

print(f"\n--- Optimal Cutoff ---")
print(f"  Cutoff Score:    {OPTIMAL_CUTOFF:.0f}")
print(f"  Approval Rate:   {opt_row['Approval_Rate']*100:.1f}%")
print(f"  Default Rate:    {opt_row['Default_Rate']*100:.2f}%")
print(f"  Net Profit:      ${opt_row['Net_Profit']:,.0f}")

print(f"\n--- Decision Rule ---")
print(f"  Score >= {OPTIMAL_CUTOFF:.0f}  →  APPROVE")
print(f"  Score <  {OPTIMAL_CUTOFF:.0f}  →  DECLINE")

print(f"\n--- Consistency Check ---")
print(f"  Scorecard range:     [{test_scores.min():.0f}, {test_scores.max():.0f}]")
print(f"  Cutoff in range:     {test_scores.min() <= OPTIMAL_CUTOFF <= test_scores.max()}")
print(f"  Headroom above:      {test_scores.max() - OPTIMAL_CUTOFF:.0f} pts")
print(f"  Headroom below:      {OPTIMAL_CUTOFF - test_scores.min():.0f} pts")